In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [2]:
dt = pd.read_csv('data.csv')
dt = pd.DataFrame(dt, columns=['MCB', 'Ngày công bố', 'Nhóm tuổi','Nghề nghiệp' ,'Xã/Phường', 'Quận/Huyện', 'Tỉnh/TP ĐK thường trú', 'G-Related-Cases', 'Ngày cách ly', 'Ngày khởi phát', 'Họ và tên'])

In [3]:
end_node = list(dt['MCB'])
name = list(dt['Họ và tên'])
occupation = list(dt['Nghề nghiệp'])

commune = list(dt['Xã/Phường'])
district = list(dt['Quận/Huyện'])
province = list(dt['Tỉnh/TP ĐK thường trú'])

group = list(map(lambda x: x.split('.')[0], list(dt['Nhóm tuổi'])))

ngay_cong_bo = list(dt['Ngày công bố'])
quarantine_day = list(map(lambda x: x.replace(" ",""), dt['Ngày cách ly']))

start_date = list(dt['Ngày khởi phát'])
start_node = list(dt['G-Related-Cases'])

In [4]:
for i in range(len(start_node)):
    start_node[i] = str(start_node[i]).replace(" ","").split(",")

In [5]:
for i in range(len(ngay_cong_bo)):
    if pd.isna(start_date[i]):
        ngay_cong_bo[i] = datetime.strptime(ngay_cong_bo[i],'%d/%m/%Y')
        quarantine_day[i] = datetime.strptime(quarantine_day[i],'%d/%m/%Y')
        d = ngay_cong_bo[i]
        start_date[i] = d - timedelta(days=5)
    else:
        try:
            start_date[i] = datetime.strptime(start_date[i], '%d/%m/%Y')
            ngay_cong_bo[i] = datetime.strptime(ngay_cong_bo[i],'%d/%m/%Y')
            quarantine_day[i] = datetime.strptime(quarantine_day[i],'%d/%m/%Y')
        except:
            print(i)
            pass

In [6]:
graph = {}

In [7]:
for i in range(len(end_node)):
    graph[end_node[i]] = [start_node[i], group[i], start_date[i], quarantine_day[i], ngay_cong_bo[i], name[i], commune[i], district[i], province[i]]

In [8]:
r_type = pd.read_csv('relation.csv')
r_type = pd.DataFrame(r_type, columns=['Relationship']).values.tolist()

In [9]:
rela = []
for src, rel in zip(graph.keys(), r_type):
    t = graph[src]
    for i in range(len(t[0])):
        rela.append([t[0][i], src, t[1], t[2], t[3], t[4], t[5], t[6], t[7], t[8], rel[0]])

In [10]:
t = pd.DataFrame(rela, columns=['source', 'target','age_group', 'onset_date', 'quarantine_date', 'announce_date', 'name', 'commune', 'district', 'province','r_type'])
t.to_csv('relation_new.csv')

In [11]:
t

,source,target,age_group,onset_date,quarantine_date,announce_date,name,commune,district,province,r_type
0,0,BN416,3,2020-07-17,2020-07-20,2020-07-25,Trần Văn Dũng,Hòa Khánh Bắc,Liên Chiểu,Đà Nẵng,0
1,0,BN418,4,2020-07-11,2020-07-18,2020-07-26,Nguyễn Văn Nam,Thanh Bình,Hải Châu,Đà Nẵng,0
2,0,BN419,1,2020-07-20,2020-07-24,2020-07-26,Ngô Bùi Trương Vũ,Quảng Phú,TP. Quảng Ngãi,Quảng Ngãi,0
3,0,BN420,4,2020-07-12,2020-07-22,2020-07-26,Phạm Thị Bích Phương,Hòa Cường Nam,Hải Châu,Đà Nẵng,0
4,0,BN421,2,2020-07-23,2020-07-25,2020-07-27,Lê Đình Huy,Tam Thuận,Thanh Khê,Đà Nẵng,0
...,...,...,...,...,...,...,...,...,...,...,...
681,BN774,BN842,1,2020-07-26,2020-08-01,2020-08-10,Nguyễn Nhật Minh,Tân An,Hội An,Quảng Nam,3
682,BN800,BN843,4,2020-08-05,2020-08-08,2020-08-10,Trần Thị Nga,Tân Chính,Thanh Khê,Đà Nẵng,3
683,BN845,BN844,2,2020-08-05,2020-07-26,2020-08-10,Nguyễn Thị Diễm Phương,Hòa Phát,Cẩm Lệ,Đà Nẵng,3
684,BN844,BN845,2,2020-08-05,2020-07-26,2020-08-10,Hoàng Thị Tâm Thư,Hòa Thuận Đông,Hải Châu,Đà Nẵng,3


## Sinh dữ liệu cho các nodes về địa điểm

In [58]:
loc_rel = []
for c, d, p in zip(commune, district, province):
    loc_rel.append([c, d, p])
    
y = pd.DataFrame(loc_rel, columns=['commune', 'district', 'province'])
y.to_csv('location.csv')